# 05 — Supervised Models

**Purpose:** Train and evaluate supervised ML models for predicting triple-barrier labels.

Key design choices:
- **Strictly chronological split** — no shuffle, no look-ahead
- **Walk-forward validation** with purge + embargo
- **Financial metrics** alongside classification metrics
- All models use the same splits for fair comparison

Models trained:
1. Logistic Regression (linear baseline)
2. Random Forest
3. XGBoost
4. LightGBM
5. CatBoost

**Inputs:** `data/features/{symbol}_labeled_*.parquet`  
**Outputs:** `models/` (saved models), feature importance DataFrames

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_paths, get_validation_params
from src.models.train_ml import (
    train_all_models, summarise_model_results, walk_forward_splits
)

cfg      = load_config()
paths    = get_paths(cfg, "..")
symbol   = get_symbol(cfg)
val_cfg  = get_validation_params(cfg)

print("Validation config:", val_cfg)

## 1. Load labeled dataset

In [ ]:
# Load baseline labeled dataset
TP_ATR, SL_ATR, HORIZON = 2.0, 1.0, 30
key      = f"tp{TP_ATR}_sl{SL_ATR}_h{HORIZON}"
lbl_path = paths["features"] / f"{symbol}_labeled_{key}.parquet"
labeled  = pd.read_parquet(lbl_path)

print(f"Loaded: {labeled.shape}")
print(f"Long label counts:\n{labeled['label_long'].value_counts().sort_index()}")

## 2. Prepare feature matrix

In [ ]:
# Exclude raw OHLCV and label columns from features
exclude_prefixes = ("open", "high", "low", "close", "tick_volume", "real_volume", "spread",
                    "label_", "barrier_side_", "time_to_exit_", "mfe_", "mae_")

feature_cols = [
    c for c in labeled.columns
    if not any(c.startswith(p) for p in exclude_prefixes)
]
print(f"Feature columns: {len(feature_cols)}")

X = labeled[feature_cols].copy()
y_long  = labeled["label_long"].copy()
y_short = labeled["label_short"].copy()

# Drop rows with NaN in features
valid_mask = X.notna().all(axis=1)
X      = X[valid_mask]
y_long  = y_long[valid_mask]
y_short = y_short[valid_mask]

# Fill remaining NaN with 0 (safety)
X = X.fillna(0)

print(f"Samples after NaN drop: {len(X)}")
print(f"Long signal rate: {(y_long == 1).mean()*100:.1f}%")

## 3. Train all models (Long labels)

In [ ]:
print("Training models on LONG labels...")
results_long = train_all_models(X, y_long, val_cfg)

summary_long = summarise_model_results(results_long)
print("\nLong model comparison:")
display(summary_long.sort_values("f1", ascending=False))

In [ ]:
print("Training models on SHORT labels...")
results_short = train_all_models(X, y_short, val_cfg)

summary_short = summarise_model_results(results_short)
print("\nShort model comparison:")
display(summary_short.sort_values("f1", ascending=False))

## 4. Feature importance

In [ ]:
# Plot feature importance for best model
best_model = summary_long.sort_values("f1", ascending=False).index[0]
fi = results_long[best_model].get("feature_importance")

if fi is not None:
    top_n = 25
    top_fi = fi.head(top_n)
    
    plt.figure(figsize=(10, 8))
    top_fi.sort_values().plot.barh(color="steelblue", alpha=0.8)
    plt.title(f"Feature Importance — {best_model} (Top {top_n})")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig("../reports/feature_importance.png", dpi=120)
    plt.show()
    
    # Save importance table
    fi.reset_index().rename(columns={"index": "feature", 0: "importance"}).to_csv(
        f"../reports/{best_model}_feature_importance.csv", index=False
    )
else:
    print("Feature importance not available for", best_model)

## 5. Save best models

In [ ]:
from src.models.train_ml import build_models, walk_forward_splits
from sklearn.base import clone

models_dir = paths["models"]
models_dir.mkdir(parents=True, exist_ok=True)

all_models = build_models()
splits = walk_forward_splits(
    len(X),
    n_splits=val_cfg["n_splits"],
    train_pct=val_cfg["train_pct"],
    purge_bars=val_cfg["purge_bars"],
    embargo_bars=val_cfg["embargo_bars"],
)

# Retrain on final fold (last train window) for deployment
if splits:
    final_train_idx = splits[-1][0]
    X_arr = X.values
    y_arr = y_long.values
    train_mask = y_arr[final_train_idx] != 0

    for name, model in all_models.items():
        try:
            m = clone(model)
            m.fit(X_arr[final_train_idx][train_mask], y_arr[final_train_idx][train_mask])
            save_path = models_dir / f"{symbol}_{name}_long.pkl"
            # Safe: pkl files are written and read exclusively by this user on this machine.
            # Never load .pkl files received from untrusted or external sources.
            joblib.dump(m, save_path)
            print(f"  Saved: {save_path}")
        except Exception as e:
            print(f"  {name}: failed — {e}")